# primitives

> Pure Python implementations of Scheme primitive procedures, isolated from the evaluator so closures cannot be shadowed by user Scheme definitions.

In [ ]:
#| default_exp primitives

### Primitive Procedures

Primitive procedures are built in and cannot be redefined — attempting `(define + ...)` raises a `SyntaxError`.

**Arithmetic**

| Operator | Meaning |
|---|---|
| `+` `-` `*` `/` | Variadic; `-` negates if unary, `/` inverts if unary |
| `=` `<` `>` `<=` `>=` | Numeric comparison |
| `abs` `min` `max` `expt` `sqrt` | Common numeric operations |
| `floor` `ceiling` `round` `truncate` | Rounding |
| `modulo` `remainder` | Integer division remainder (differ on negatives) |

**Lists**

| Operator | Meaning |
|---|---|
| `list` `cons` `car` `cdr` | Construction and access |
| `null?` `pair?` `list?` | Predicates |

**Strings**

| Operator | Meaning |
|---|---|
| `string-append` `string-length` `substring` | Operations |
| `number->string` `string->number` | Numeric conversion |
| `symbol->string` `string->symbol` | Symbol conversion |

**Predicates & equality**

| Operator | Meaning |
|---|---|
| `number?` `string?` `symbol?` `boolean?` | Type predicates |
| `equal?` | Deep structural equality (type-strict) |
| `not` | Boolean negation |

**Other**

| Operator | Meaning |
|---|---|
| `error` | Raise an exception with a message and optional irritants |

In [ ]:
import math, operator as op
from compact.types import *

In [ ]:
_sum = sum

In [ ]:
_primitives = {}

def primitive(name):
    def reg(fn): _primitives[name] = fn; return fn
    return reg

def ls(): return _primitives

In [ ]:
@primitive("+")
def _plus(*xs): return _sum(xs)

@primitive("*")
def _mul(*xs): return math.prod(xs)

@primitive("-")
def _scm_sub(x, *xs): return x - _sum(xs) if xs else -x

@primitive("/")
def _scm_div(x, *xs): return 1/x if not xs else x / math.prod(xs)

@primitive("=")
def _eq(x, y): return _is_num(x) and _is_num(y) and x == y

In [ ]:
for _name, _fn in [("<", op.lt), (">", op.gt), ("<=", op.le), (">=", op.ge),
    ("abs", abs), ("min", min), ("max", max), ("sqrt", math.sqrt),
    ("expt", pow), ("modulo", op.mod), ("floor", math.floor),
    ("ceiling", math.ceil), ("round", round), ("truncate", math.trunc)]:
    primitive(_name)(_fn)

In [ ]:
@primitive("remainder")
def _remainder(x, y): return x - int(x/y)*y

In [ ]:
def _is_num(x): return not isinstance(x, bool) and isinstance(x, (int, float, complex))
def _is_sym(x): return isinstance(x, Symbol)
def _is_list(x): return isinstance(x, list)
def _is_pair(x): return isinstance(x, list) and bool(x)

for _name, _fn in [("number?", _is_num), ("string?", lambda x: isinstance(x, str)),
    ("symbol?", _is_sym), ("boolean?", lambda x: isinstance(x, bool)),
    ("null?", lambda x: x == []), ("pair?", _is_pair), ("list?", _is_list)]:
    primitive(_name)(_fn)

In [ ]:
@primitive("list")
def _list(*xs): return list(xs)

@primitive("cons")
def _cons(x, y):
    if not isinstance(y, list): raise TypeError(f"cons: second argument must be a list, got {type(y).__name__}")
    return [x] + y

@primitive("car")
def _car(x): return x[0]

@primitive("cdr")
def _cdr(x): return x[1:]

In [ ]:
@primitive("equal?")
def _scm_equal(x, y):
    if type(x) != type(y): return False
    if isinstance(x, list): return len(x) == len(y) and all(_scm_equal(a,b) for a,b in zip(x,y))
    return x == y

@primitive("not")
def _not(x): return x is False

@primitive("error")
def _scm_error(msg, *irritants): raise Exception(msg if not irritants else f"{msg} {' '.join(map(repr, irritants))}")

In [ ]:
@primitive("string-append")
def _str_append(*xs): return "".join(xs)

@primitive("string-length")
def _str_len(s): return len(s)

@primitive("substring")
def _substring(s, i, j): return s[i:j]

@primitive("number->string")
def _num2str(x): return str(x)

@primitive("string->number")
def _str2num(s):
    try: return int(s)
    except ValueError:
        try: return float(s)
        except ValueError: return False

@primitive("symbol->string")
def _sym2str(x): return x.s

@primitive("string->symbol")
def _str2sym(x): return Symbol(x)

In [ ]:
@primitive("map")
def _map(fn, *lsts): return [fn(*args) for args in zip(*lsts)]

@primitive("filter")
def _filter(fn, lst): return [x for x in lst if fn(x) is not False]

@primitive("for-each")
def _scm_for_each(fn, *lsts):
    for args in zip(*lsts): fn(*args)

@primitive("fold-left")
def _scm_fold_left(fn, init, lst):
    acc = init
    for x in lst: acc = fn(acc, x)
    return acc

@primitive("fold-right")
def _scm_fold_right(fn, init, lst):
    acc = init
    for x in reversed(lst): acc = fn(x, acc)
    return acc

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()